# TimesFM 2.5 LoRA Fine-Tuning - VN30 Volatility Prediction

**Google Colab Training Notebook**

**Repository:** https://github.com/ntquy9901/stock_vol_prediction01

**Goal:** Fine-tune TimesFM 2.5 on VN30 Parkinson volatility (2006-2026)

**Hardware:** GPU (Tesla T4/A100)

**Training Time:**
- Quick test (5 epochs): 5-10 minutes
- Full training (70 epochs): 2-3 hours

## 🚀 Phase 1: Setup (5 minutes)

### Step 1: Enable GPU Runtime

1. Click **Runtime** → **Change runtime type**
2. Select **T4 GPU** (free) or **A100** (Pro)
3. Click **Save**

### Step 2: Verify GPU

Run the cell below to verify GPU is available:

In [ ]:
# Verify GPU is available
!nvidia-smi

### Step 3: Clone Repository

Clone the repository from GitHub:

In [ ]:
# Clone repository
!git clone https://github.com/ntquy9901/stock_vol_prediction01.git

# Change to repository directory
%cd stock_vol_prediction01

# Verify files are present
!ls -la
!echo "\n=== Repository structure ==="
!ls -la src/
!echo "\n=== Data files ==="
!ls data/raw/prices/ | head -10

### Step 4: Install Dependencies

Install PyTorch, Transformers, and other required packages:

In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install Transformers and PEFT
!pip install transformers==4.36.0 peft==0.7.1 accelerate==0.25.0 bitsandbytes==0.41.0

# Install other dependencies
!pip install pandas numpy scikit-learn matplotlib mlflow

# Verify installations
!echo "=== PyTorch ==="
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

---

## 🎯 Phase 2: Quick Test (5-10 minutes)

### Overview

Run a quick test with **1 stock, 5 epochs** to verify everything works.

**Expected time:** 5-10 minutes on Tesla T4

**This will:**
- Download TimesFM 2.5 model (~950 MB)
- Train for 5 epochs
- Save results to `results/timesfm_quick_test/`

In [ ]:
# Quick test - 1 stock, 5 epochs
!python -m src.timesfm_baseline.timesfm_baseline/train_timesfm_lora.py \
    --data_path data/raw/prices/ACB_ohlcv.csv \
    --stock_id ACB \
    --output_dir results/timesfm_quick_test \
    --epochs 5 \
    --batch_size 32 \
    --context_len 64 \
    --horizon_len 5 \
    --lr 1e-4 \
    --patience 15 \
    --seed 42

### Check Quick Test Results

After training completes, check the results:

In [ ]:
# View training results
!ls -la results/timesfm_quick_test/

# View final metrics
!cat results/timesfm_quick_test/final_metrics.json

# Display learning curves
from IPython.display import Image, display
import glob
import os

learning_curves = glob.glob('results/timesfm_quick_test/learning_curves_*.png')
if learning_curves:
    print(f"\n=== Learning Curves ===")
    display(Image(filename=learning_curves[0]))
else:
    print("No learning curves found")

---

## 🚀 Phase 3: Full Training (2-3 hours)

### Overview

Run full training with **all 30 stocks, 70 epochs**.

**Expected time:** 2-3 hours on Tesla T4

**This will:**
- Train on all 30 VN30 stocks
- Run for 70 epochs with early stopping
- Save checkpoints every half epoch
- Save best model automatically
- Track metrics with MLflow
- Save results to `results/timesfm_full_<timestamp>/`

In [ ]:
# Full training - All stocks, 70 epochs
!python -m src.timesfm_baseline.timesfm_baseline/train_timesfm_lora.py \
    --multi_stock \
    --data_path data/raw/prices \
    --output_dir results/timesfm_full_$(date +%Y%m%d_%H%M%S) \
    --epochs 70 \
    --batch_size 32 \
    --context_len 64 \
    --horizon_len 5 \
    --lr 1e-4 \
    --patience 15 \
    --seed 42

### Monitor Training Progress

While training is running, you can monitor progress:

In [ ]:
# Monitor training progress (watch in real-time)
!ls -la results/

# View latest training log (if exists)
latest_results = !ls -td results/timesfm_full_* | head -1
latest_results = str(latest_results).strip().strip("\"")
if latest_results:
    print(f"\n=== Latest Results Directory: {latest_results} ===\n")
    !ls -la {latest_results}
else:
    print("\nNo results found yet. Training still in progress.")

---

## 💾 Phase 4: Save Results

### Option 1: Download to Local Machine

1. Click 📁 **Files** icon in left sidebar
2. Navigate to `results/timesfm_full_<timestamp>/`
3. Right-click files → **Download**

### Option 2: Save to Google Drive (Recommended)

Mount Google Drive and copy results:

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy results to Google Drive
!mkdir -p /content/drive/MyDrive/TimesFM_Results/

# Copy all results directories
latest_results = !ls -td results/timesfm_full_* | head -1
latest_results = str(latest_results).strip().strip("\"")
if latest_results:
    !cp -r {latest_results} /content/drive/MyDrive/TimesFM_Results/
    print(f"\n✅ Results copied to Google Drive: /content/drive/MyDrive/TimesFM_Results/")
    !ls -la /content/drive/MyDrive/TimesFM_Results/
else:
    print("\n❌ No results found to copy.")

### Download Best Model

Download the trained LoRA adapters:

In [ ]:
# Zip and download best model
import os
import zipfile
from google.colab import files

# Find best model directory
latest_results = !ls -td results/timesfm_full_* | head -1
latest_results = str(latest_results).strip().strip("\"")
if latest_results:
    best_model_dir = f"{latest_results}/best_model_*/"
    
    # Create zip file
    zip_filename = "best_model.zip"
    
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for root, dirs, files in os.walk(best_model_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, latest_results)
                zipf.write(file_path, arcname)
    
    print(f"\n✅ Best model zipped: {zip_filename}")
    print(f"Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")
    
    # Download zip file
    files.download(zip_filename)
else:
    print("\n❌ No trained model found.")

---

## 📊 Phase 5: Evaluate Model

### Load Trained Model and Make Predictions

In [ ]:
# Load trained model for prediction
from src.timesfm_baseline.timesfm_lora_finetuning import TimesFMLoRAFineTuner
import pandas as pd
import torch
import numpy as np

# Initialize finetuner
finetuner = TimesFMLoRAFineTuner(
    context_len=64,
    horizon_len=5,
    device='cuda'
)

# Find best model path
latest_results = !ls -td results/timesfm_full_* | head -1
latest_results = str(latest_results).strip().strip("\"")
if latest_results:
    # Load trained model
    finetuner.load_model(
        model_id='google/timesfm-2.5-500m',
        adapter_path=f"{latest_results}/best_model_*/"
    )
    
    print("✅ Model loaded successfully!")
else:
    print("❌ No trained model found. Please run training first.")

In [ ]:
# Make predictions on test data
import pandas as pd
from src.common.parkinson_utils import calculate_parkinson_volatility
from src.timesfm_baseline.volatility_dataset import TimeSeriesLastWindowDataset

# Load test data
test_data = pd.read_csv('data/raw/prices/VIC_ohlcv.csv')
test_volatility = calculate_parkinson_volatility(test_data)

# Create dataset
series = test_volatility['parkinson_volatility'].values
dataset = TimeSeriesLastWindowDataset([series], context_len=64, horizon_len=5)

# Get last window for prediction
if len(dataset) > 0:
    context, target = dataset[0]
    context = context.unsqueeze(0).to('cuda')  # Add batch dimension
    
    # Make prediction
    with torch.no_grad():
        output = finetuner.model(
            past_values=context,
            forecast_context_len=64
        )
        prediction = output.mean_predictions[0, :5].cpu().numpy()
    
    print("\n=== 5-Day Volatility Forecast ===")
    print(f"Day 1: {prediction[0]:.6f}")
    print(f"Day 2: {prediction[1]:.6f}")
    print(f"Day 3: {prediction[2]:.6f}")
    print(f"Day 4: {prediction[3]:.6f}")
    print(f"Day 5: {prediction[4]:.6f}")
    
    # Plot prediction
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 6))
    
    # Historical volatility
    historical = series[-100:]
    plt.plot(range(len(historical)), historical, label='Historical', linewidth=2)
    
    # Forecast
    forecast_x = range(len(historical), len(historical) + 5)
    plt.plot(forecast_x, prediction, 'ro-', label='Forecast', linewidth=2, markersize=8)
    
    plt.xlabel('Day')
    plt.ylabel('Parkinson Volatility')
    plt.title('5-Day Volatility Forecast')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("❌ Not enough data for prediction")

---

## 💡 Tips & Best Practices

### 1. Prevent Session Timeout

Colab sessions timeout after 90 minutes of inactivity. To prevent this:

**Option A: Run in background**
```python
# Run training in background
!nohup python src/timesfm_baseline/train_timesfm_lora.py \
    --multi_stock --data_path data/raw/prices \
    --output_dir results --epochs 70 > training.log 2>&1 &
```

**Option B: Use JavaScript console**
- Press F12 to open browser console
- Paste: `function ClickConnect(){console.log("Working...");document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)`

### 2. Monitor GPU Usage

```python
# Check GPU memory usage
!nvidia-smi

# Check GPU memory in PyTorch
import torch
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
```

### 3. Handle Out-of-Memory Errors

If you get CUDA OOM error:

**Solution 1: Reduce batch size**
```python
!python -m src.timesfm_baseline.timesfm_baseline/train_timesfm_lora.py --batch_size 16 ...
```

**Solution 2: Reduce context length**
```python
!python -m src.timesfm_baseline.timesfm_baseline/train_timesfm_lora.py --context_len 32 ...
```

**Solution 3: Clear GPU cache**
```python
import torch
torch.cuda.empty_cache()
```

### 4. Save Results Frequently

- Use Google Drive option (recommended)
- Checkpoints are saved automatically every half epoch
- Best model is saved automatically
- Learning curves are plotted every 10 epochs

### 5. Troubleshooting Common Issues

**Issue: Model download fails**
- Solution: Use huggingface-cli login

**Issue: Training is too slow**
- Solution: Verify GPU is enabled (!nvidia-smi)

**Issue: Colab session disconnects**
- Solution: Use Google Drive to save results

**Issue: Data not found**
- Solution: Verify repository was cloned correctly (!ls data/raw/prices/)

---

## 📚 Summary

### What You've Done

✅ Cloned TimesFM repository from GitHub
✅ Installed all dependencies (PyTorch, Transformers, PEFT)
✅ Verified GPU is available and working
✅ Ran quick test (5 epochs) to verify setup
✅ Ran full training (70 epochs) on all 30 VN30 stocks
✅ Saved results to Google Drive
✅ Downloaded trained model
✅ Made predictions on test data

### Next Steps

1. **Pull results to local machine:**
   ```bash
   git pull origin master
   ```

2. **Evaluate model performance:**
   - Compare with baseline models (HAR, LSTM)
   - Analyze prediction errors
   - Visualize attention weights

3. **Improve model:**
   - Tune hyperparameters
   - Try different context lengths
   - Experiment with ensemble methods

### Resources

- **GitHub Repository:** https://github.com/ntquy9901/stock_vol_prediction01
- **Full Documentation:** See `docs/` folder in repository
- **Lessons Learned:** `docs/LESSONS_LEARNED_TIMESFM_ADVERSARIAL_REVIEWS.md`
- **Troubleshooting:** `docs/GITHUB_SETUP_GUIDE.md`

### Model Performance Targets

| Metric | Baseline | Target | Status |
|--------|----------|--------|--------|
| RMSE | 0.18 | < 0.15 | 🎯 Training |
| Dir Acc | 55% | > 60% | 🎯 Training |
| Training Time | 8-10h (CPU) | 2-3h (GPU) | ✅ Achieved |

---

**Happy Training! 🚀**

**Last Updated:** 2026-06-20
**Repository:** https://github.com/ntquy9901/stock_vol_prediction01